In [ ]:
from time import perf_counter
import cv2
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass


IMAGE = "/content/AlexClark-full.jpg"
TIMINGS = {}

def tic(name):
    TIMINGS[name] = -perf_counter()

def toc(name):
    TIMINGS[name] += perf_counter()


In [ ]:
# @title
# ============================================================
# NuThing P1 Lab v2 — masks + components (canonical, unchanged)
#
# Same bright-neutral/dark masks and PCA component geometry as the
# original canonical prototype. Everything downstream is replaced by
# render-identity detectors (cells 2-4): the render stack draws EXACT
# repeated primitives, so P1 should match those primitives instead of
# clustering generic component families.
#
# Session-measured facts this notebook is built on (dev corpus, overview
# zoom; branch claude/nuthing-p2-digit-recognition-zgs4lq):
#   * the basket is ONE byte-stable 42x66 sprite -> matched filter,
#     occlusion-tolerant, 72/72 dev truth baskets;
#   * the tee pad is the stack's only small HOLLOW glyph (white ring,
#     dark interior, rotated) -> enclosed-hole detection, 69/72, with
#     diamond ring/path markers split off by interior elongation;
#   * the tee long axis points at its own hole's badge
#     (median 1.1 deg, p90 2.65; false badges median 38 deg).
# ============================================================

BRIGHT_V_MIN = 210
BRIGHT_S_MAX = 45
DARK_V_MAX = 45
MIN_COMPONENT_AREA = 12

# Badge frames are screen-aligned, white-bordered, dark inside.
BADGE_ASPECT_MIN = 1.15
BADGE_ASPECT_MAX = 1.80
BADGE_MIN_DARK_INTERIOR = 0.45
BADGE_SIZE_TOL = np.log(1.15)
FAMILY_SIZE_TOL = np.log(1.20)


@dataclass
class Component:
    label: int
    mask_kind: str
    cx: float
    cy: float
    area: int
    bbox_x: int
    bbox_y: int
    bbox_w: int
    bbox_h: int
    major: float
    minor: float
    angle_deg: float
    fill: float

    @property
    def aspect(self):
        return self.major / max(self.minor, 1e-6)


tic("masks")
bgr = cv2.imread(IMAGE, cv2.IMREAD_COLOR)
if bgr is None:
    raise ValueError(f"Could not read {IMAGE}")
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
s = hsv[:, :, 1]
v = hsv[:, :, 2]
bright_mask = ((v >= BRIGHT_V_MIN) & (s <= BRIGHT_S_MAX)).astype(np.uint8)
dark_mask = (v <= DARK_V_MAX).astype(np.uint8)
toc("masks")


def pca_geometry(xs, ys):
    pts = np.column_stack([xs, ys]).astype(np.float64)
    if len(pts) < 2:
        return 1.0, 1.0, 0.0
    centered = pts - pts.mean(axis=0)
    cov = np.cov(centered, rowvar=False)
    if cov.shape != (2, 2):
        return 1.0, 1.0, 0.0
    values, vectors = np.linalg.eigh(cov)
    order = np.argsort(values)[::-1]
    vectors = vectors[:, order]
    pj_major = centered @ vectors[:, 0]
    pj_minor = centered @ vectors[:, 1]
    major = pj_major.max() - pj_major.min() + 1
    minor = pj_minor.max() - pj_minor.min() + 1
    angle_deg = np.degrees(np.arctan2(vectors[1, 0], vectors[0, 0])) % 180
    if minor > major:
        major, minor = minor, major
        angle_deg = (angle_deg + 90) % 180
    return float(major), float(minor), float(angle_deg)


def extract_components(binary, mask_kind):
    count, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary, connectivity=8)
    components = []
    for label in range(1, count):
        x, y, w, h, area = stats[label]
        if area < MIN_COMPONENT_AREA:
            continue
        roi = labels[y:y + h, x:x + w]
        local_ys, local_xs = np.where(roi == label)
        major, minor, angle_deg = pca_geometry(local_xs + x, local_ys + y)
        components.append(Component(
            label=label, mask_kind=mask_kind,
            cx=float(centroids[label][0]), cy=float(centroids[label][1]),
            area=int(area),
            bbox_x=int(x), bbox_y=int(y), bbox_w=int(w), bbox_h=int(h),
            major=major, minor=minor, angle_deg=angle_deg,
            fill=area / max(major * minor, 1),
        ))
    return components, labels


tic("components")
bright_components, bright_labels = extract_components(bright_mask, "bright")
dark_components, _ = extract_components(dark_mask, "dark")
toc("components")


# --- badge family (kept from the canonical prototype: badges were never
# --- the weak stage) -------------------------------------------------
def interior_dark_fraction(c):
    x0, y0 = c.bbox_x + 2, c.bbox_y + 2
    x1, y1 = c.bbox_x + c.bbox_w - 2, c.bbox_y + c.bbox_h - 2
    if x1 <= x0 or y1 <= y0:
        return 0.0
    roi = dark_mask[y0:y1, x0:x1]
    return float(roi.mean())


badge_pool = [
    c for c in bright_components
    if BADGE_ASPECT_MIN <= (c.bbox_w / max(c.bbox_h, 1)) <= BADGE_ASPECT_MAX
    and interior_dark_fraction(c) >= BADGE_MIN_DARK_INTERIOR
    and 14 <= c.bbox_h <= 44
]
badge_families = []
used = set()
for seed in sorted(badge_pool, key=lambda c: -c.area):
    if seed.label in used:
        continue
    fam = [c for c in badge_pool if c.label not in used and
           abs(np.log(max(c.bbox_w, 1) / max(seed.bbox_w, 1))) <= BADGE_SIZE_TOL and
           abs(np.log(max(c.bbox_h, 1) / max(seed.bbox_h, 1))) <= BADGE_SIZE_TOL]
    for c in fam:
        used.add(c.label)
    badge_families.append(fam)
badge_families.sort(key=len, reverse=True)
badges = badge_families[0] if badge_families else []
print(f"bright components: {len(bright_components)}   "
      f"dark components: {len(dark_components)}   badges: {len(badges)}")


In [ ]:
# @title
# ============================================================
# Basket sprites — occlusion-tolerant matched filter
#
# The basket is one fixed, unrotated sprite bitmap repeated at one scale
# per screenshot (byte-identical in 60/66 clean dev detections). So:
#   1. build the consensus sprite bitmap FROM THIS IMAGE (majority vote
#      over the modal clean-sprite family);
#   2. matched-filter the bright mask: score = onFrac - offFrac.
#      A solid white blob scores 0 (onFrac == offFrac == 1); a clean
#      sprite ~0.95; a half-occluded one ~0.3.
#   3. matching-pursuit dedupe: accept the best match, ERASE the pixels
#      it explains, re-score, repeat. A shifted echo of an accepted
#      sprite collapses; a genuinely overlapping neighbor sprite keeps
#      its own pixels and survives. Radius-NMS cannot make that
#      distinction (real neighbors sit closer than a sprite diagonal).
#
# Basket ENDPOINT = pole tip = sprite bottom + 4 px
# (BASKET_SPRITE_TIP_OFFSET_PX).
# ============================================================

SPRITE_SCORE_MIN = 0.28
TIP_OFFSET = 4

tic("sprite_template")
# Clean sprite family: near-exact 42x66 bright components.
clean = [c for c in bright_components
         if 40 <= c.bbox_w <= 44 and 62 <= c.bbox_h <= 70
         and c.area >= 1500 and c.fill >= 0.5]
if len(clean) < 2:
    raise ValueError(
        "No clean 42x66 sprite family found — check zoom level / crop. "
        f"(candidates: {len(clean)})")
TW = int(np.median([c.bbox_w for c in clean]))
TH = int(np.median([c.bbox_h for c in clean]))
votes = np.zeros((TH, TW), np.float32)
n_votes = 0
for c in clean:
    if c.bbox_w != TW or c.bbox_h != TH:
        continue
    roi = bright_labels[c.bbox_y:c.bbox_y + TH, c.bbox_x:c.bbox_x + TW]
    votes += (roi == c.label).astype(np.float32)
    n_votes += 1
template = (votes >= n_votes / 2).astype(np.uint8)
print(f"sprite template {TW}x{TH} from {n_votes} clean detections, "
      f"on-pixels={int(template.sum())}")
toc("sprite_template")

tic("sprite_match")
on_k = template.astype(np.float32)
off_k = (1 - template).astype(np.float32)
n_on = float(on_k.sum())
n_off = float(off_k.sum())


def sprite_scores(mask):
    f = mask.astype(np.float32)
    on_hits = cv2.filter2D(f, -1, on_k / n_on, anchor=(0, 0),
                           borderType=cv2.BORDER_CONSTANT)
    off_hits = cv2.filter2D(f, -1, off_k / n_off, anchor=(0, 0),
                            borderType=cv2.BORDER_CONSTANT)
    # anchor=(0,0): score[y,x] corresponds to template top-left at (x,y)
    return on_hits - off_hits


live = bright_mask.copy()
sprites = []  # dicts: x, y (top-left), cx, cy, tip_x, tip_y, score
scores = sprite_scores(live)
H, W_img = live.shape
for _ in range(200):
    y, x = np.unravel_index(np.argmax(scores), scores.shape)
    sc = float(scores[y, x])
    if sc < SPRITE_SCORE_MIN:
        break
    if y + TH > H or x + TW > W_img:
        scores[y, x] = -1
        continue
    sprites.append(dict(
        x=int(x), y=int(y),
        cx=x + TW / 2, cy=y + TH / 2,
        tip_x=x + TW / 2, tip_y=y + TH + TIP_OFFSET,
        score=sc,
    ))
    # erase the claimed pixels, re-score locally (2 template spans around)
    live[y:y + TH, x:x + TW] &= (1 - template)
    y0, y1 = max(0, y - TH), min(H, y + 2 * TH)
    x0, x1 = max(0, x - TW), min(W_img, x + 2 * TW)
    pad_y0, pad_x0 = max(0, y0 - TH), max(0, x0 - TW)
    local = sprite_scores(live[pad_y0:y1 + TH, pad_x0:x1 + TW])
    scores[y0:y1, x0:x1] = local[y0 - pad_y0:y1 - pad_y0,
                                 x0 - pad_x0:x1 - pad_x0]
toc("sprite_match")

sprites.sort(key=lambda sp: -sp["score"])
print(f"basket sprites: {len(sprites)} "
      f"(scores {sprites[-1]['score']:.2f}..{sprites[0]['score']:.2f})"
      if sprites else "basket sprites: 0")
for sp in sprites:
    print(f"  tip=({sp['tip_x']:6.1f},{sp['tip_y']:6.1f}) score={sp['score']:.2f}")


In [ ]:
# @title
# ============================================================
# Tee pads — hollow-ring detection (+ component fallback)
#
# The tee pad is the render stack's only small HOLLOW glyph: a thick
# white rect outline around transparent (basemap-dark) interior, rotated
# to the throwing direction. Detection:
#   * flood the non-bright background in from the border; any remaining
#     non-bright pixel is a hole fully enclosed by bright pixels;
#   * flood against DILATED walls at radius 0/1/2/3 to seal the 1-4 px
#     outline gaps occlusion cuts, but always MEASURE the hole on the
#     raw mask (morphological closing fills small holes outright);
#   * verify the enclosing ring band on the raw mask;
#   * interior elongation splits tee rects (~2:1) from diamond ring/path
#     markers (~1:1) — diamonds are kept, tagged, as distractors;
#   * a component-family fallback covers rings whose gap opens into a
#     C2D circle (structurally unclosable).
#
# The hole's principal axis is the tee ORIENTATION — it points at the
# hole's own badge (median 1.1 deg on dev truth). Do NOT exclude
# candidates for standing on a C2D ring: real tees do exactly that.
# ============================================================

HOLE_AREA_MIN = 10
HOLE_AREA_MAX = 480
HOLE_DIM_MAX = 44
RING_BAND = 3
RING_FRAC_MIN = 0.6
TEE_ELONGATION_MIN = 1.18


def detect_rings_pass(wall_radius, area_min):
    if wall_radius > 0:
        kernel = np.ones((3, 3), np.uint8)
        wall = cv2.dilate(bright_mask, kernel, iterations=wall_radius)
    else:
        wall = bright_mask
    # background flood from the border over non-wall pixels
    free = (1 - wall).astype(np.uint8)
    ff = free.copy()
    h, w = ff.shape
    ffmask = np.zeros((h + 2, w + 2), np.uint8)
    for seed in [(0, 0), (w - 1, 0), (0, h - 1), (w - 1, h - 1)]:
        if ff[seed[1], seed[0]]:
            cv2.floodFill(ff, ffmask, seed, 2)
    # border rows/cols may have multiple disconnected free regions
    for x in range(w):
        for y in (0, h - 1):
            if ff[y, x] == 1:
                cv2.floodFill(ff, ffmask, (x, y), 2)
    for y in range(h):
        for x in (0, w - 1):
            if ff[y, x] == 1:
                cv2.floodFill(ff, ffmask, (x, y), 2)
    holes_mask = ((ff == 1) & (bright_mask == 0)).astype(np.uint8)
    count, labels, stats, cents = cv2.connectedComponentsWithStats(
        holes_mask, connectivity=4)
    out = []
    for lb in range(1, count):
        x, y, bw, bh, area = stats[lb]
        if area < area_min or area > HOLE_AREA_MAX:
            continue
        if bw > HOLE_DIM_MAX or bh > HOLE_DIM_MAX:
            continue
        roi = (labels[y:y + bh, x:x + bw] == lb).astype(np.uint8)
        ys, xs = np.nonzero(roi)
        cx, cy = float(cents[lb][0]), float(cents[lb][1])
        major, minor, angle = pca_geometry(xs + x, ys + y)
        elong = major / max(minor, 1e-6)
        # ring band: bright fraction within RING_BAND of the hole
        kernel = np.ones((3, 3), np.uint8)
        dil = cv2.dilate(roi, kernel, iterations=RING_BAND)
        band_local = dil - roi
        y0, x0 = max(0, y - RING_BAND), max(0, x - RING_BAND)
        pad = np.zeros((bh + 2 * RING_BAND, bw + 2 * RING_BAND), np.uint8)
        pad[RING_BAND:RING_BAND + bh, RING_BAND:RING_BAND + bw] = roi
        dil2 = cv2.dilate(pad, kernel, iterations=RING_BAND)
        band = dil2 - pad
        bys, bxs = np.nonzero(band)
        vals = []
        for by, bx in zip(bys, bxs):
            gy, gx = y - RING_BAND + by, x - RING_BAND + bx
            if 0 <= gy < h and 0 <= gx < w:
                vals.append(bright_mask[gy, gx])
        ring_frac = float(np.mean(vals)) if vals else 0.0
        if ring_frac < RING_FRAC_MIN:
            continue
        out.append(dict(cx=cx, cy=cy, area=int(area), elong=elong,
                        angle=angle, ring_frac=ring_frac,
                        kind="tee-rect" if elong >= TEE_ELONGATION_MIN
                        else "diamond"))
    return out


tic("tee_rings")
rings = []
for radius in (0, 1, 2, 3):
    area_min = 40 if radius >= 2 else HOLE_AREA_MIN
    for r in detect_rings_pass(radius, area_min):
        if any(np.hypot(m["cx"] - r["cx"], m["cy"] - r["cy"]) < 10
               for m in rings):
            continue
        rings.append(r)
toc("tee_rings")

badge_labels = {c.label for c in badges}


def inside_badge(x, y):
    return any(c.bbox_x - 3 <= x <= c.bbox_x + c.bbox_w + 3 and
               c.bbox_y - 3 <= y <= c.bbox_y + c.bbox_h + 3
               for c in badges)


ring_tees = [r for r in rings if r["kind"] == "tee-rect"
             and not inside_badge(r["cx"], r["cy"])]
diamonds = [r for r in rings if r["kind"] == "diamond"
            and not inside_badge(r["cx"], r["cy"])]

# fallback tier: widened tee-rect component family, minus anything
# already explained (ring tees, diamonds, sprites)
tic("tee_fallback")
tees = [dict(cx=r["cx"], cy=r["cy"], tier="ring", angle=r["angle"],
             elong=r["elong"]) for r in ring_tees]
for c in bright_components:
    if c.label in badge_labels or inside_badge(c.cx, c.cy):
        continue
    mn, mx = min(c.bbox_w, c.bbox_h), max(c.bbox_w, c.bbox_h)
    if mn < 8 or mx > 42 or not (80 <= c.area <= 350)        or not (0.2 <= c.fill <= 0.85):
        continue
    if any(np.hypot(t["cx"] - c.cx, t["cy"] - c.cy) < 12 for t in tees):
        continue
    if any(np.hypot(d["cx"] - c.cx, d["cy"] - c.cy) < 12 for d in diamonds):
        continue
    if any(np.hypot(sp["cx"] - c.cx, sp["cy"] - c.cy) < 24 for sp in sprites):
        continue
    tees.append(dict(cx=c.cx, cy=c.cy, tier="component", angle=None,
                    elong=None))
toc("tee_fallback")

print(f"tees: {len(tees)} "
      f"({sum(1 for t in tees if t['tier'] == 'ring')} ring / "
      f"{sum(1 for t in tees if t['tier'] == 'component')} component), "
      f"diamond markers: {len(diamonds)}")


In [ ]:
# @title
# ============================================================
# Tee-border verification — the DELIBERATE metric, ring-seeded
#
# The projected-border ranking from the original P1 lab, verbatim in its
# scoring core: learn a fuzzy consensus border from 3 seed tees in a
# shared canonical frame (ONE scale from the seed family — a chrome
# fragment cannot become tee-sized through normalization), then rank
# every candidate by F-beta(0.5) agreement: precision favored (observed
# pixels should look like tee border), missing border tolerated more —
# which is exactly what lets partially occluded pads still rank.
# ============================================================

SEED_COUNT = 3

CANON_SIZE = 96
CANON_MAJOR_SPAN = 60.0

# Border-match softness in canonical pixels.
MATCH_SIGMA = 1.75

# Small translation search protects against centroid drift caused
# by partial/occluded tees.
SHIFT_VALUES = [-4, -2, 0, 2, 4]

# Precision is deliberately favored:
# observed pixels should look like tee border.
# Missing expected border is tolerated more.
F_BETA = 0.5

# Exact T5 truth from annotation.
T5_X = 483.59288537549406
T5_Y = 852.9288537549407


labels = bright_labels


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def size_distance(c, major, minor):
    return max(
        abs(np.log(c.major / major)),
        abs(np.log(c.minor / minor))
    )


def get_component_pixels(c):
    """
    Recover exact pixels belonging to this component.

    Uses the same bright-mask connected-component labeling as P1.
    """
    x0 = int(c.bbox_x)
    y0 = int(c.bbox_y)
    w = int(c.bbox_w)
    h = int(c.bbox_h)

    roi = labels[y0:y0+h, x0:x0+w]

    yy, xx = np.where(roi == c.label)

    # Defensive fallback in case label numbering somehow differs.
    if len(xx) == 0:
        roi_labels = roi[roi > 0]

        if len(roi_labels) == 0:
            return np.array([]), np.array([])

        values, counts = np.unique(roi_labels, return_counts=True)
        best_label = values[np.argmax(counts)]

        yy, xx = np.where(roi == best_label)

    return xx + x0, yy + y0


def canonical_component_mask(c, scale):
    """
    Rotate component into PCA coordinates and place it into a
    shared canonical frame.

    IMPORTANT:
    We use ONE shared scale derived from the modal tee family.

    We do NOT stretch every candidate to fit the template.
    Therefore a weirdly sized chrome fragment cannot become
    tee-sized merely through normalization.
    """
    xs, ys = get_component_pixels(c)

    out = np.zeros(
        (CANON_SIZE, CANON_SIZE),
        dtype=np.uint8
    )

    if len(xs) < 2:
        return out

    pts = np.column_stack([xs, ys]).astype(np.float64)

    center = pts.mean(axis=0)
    rel = pts - center

    cov = np.cov(rel, rowvar=False, bias=True)

    eigvals, eigvecs = np.linalg.eigh(cov)

    # Largest eigenvector = major axis.
    major_axis = eigvecs[:, np.argmax(eigvals)]

    # Deterministic sign.
    if (
        major_axis[0] < 0
        or (
            abs(major_axis[0]) < 1e-9
            and major_axis[1] < 0
        )
    ):
        major_axis = -major_axis

    minor_axis = np.array([
        -major_axis[1],
         major_axis[0]
    ])

    u = rel @ major_axis
    v = rel @ minor_axis

    cx = (CANON_SIZE - 1) / 2
    cy = (CANON_SIZE - 1) / 2

    px = np.rint(cx + u * scale).astype(int)
    py = np.rint(cy + v * scale).astype(int)

    valid = (
        (px >= 0) &
        (px < CANON_SIZE) &
        (py >= 0) &
        (py < CANON_SIZE)
    )

    out[py[valid], px[valid]] = 1

    return out


def distance_to_foreground(mask):
    """
    For every pixel, distance to nearest foreground pixel.
    """
    inverse = (mask == 0).astype(np.uint8)

    return cv2.distanceTransform(
        inverse,
        cv2.DIST_L2,
        3
    )


def fuzzy_support(dist, sigma=MATCH_SIGMA):
    return np.exp(
        -(dist ** 2) /
        (2.0 * sigma ** 2)
    )


def shift_mask(mask, dx, dy):
    M = np.float32([
        [1, 0, dx],
        [0, 1, dy]
    ])

    return cv2.warpAffine(
        mask,
        M,
        (CANON_SIZE, CANON_SIZE),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )


def fbeta(precision, recall, beta=F_BETA):
    if precision <= 0 or recall <= 0:
        return 0.0

    b2 = beta * beta

    return (
        (1 + b2) *
        precision *
        recall
        /
        (b2 * precision + recall)
    )


# ------------------------------------------------------------
# 1. Find modal tee population
# ------------------------------------------------------------
# 1-3. Seeds come from the hollow-ring detector (cell 3), not from
# modal-family clustering. MEASURED WHY (dev corpus): with modal-family
# seeds this metric is perfect on DashsTrack (all 18 truth tees in the
# top 18) but collapses on Heritage (0/18) because family clustering
# there picks the walking-dash family and the learned border is a dash
# border. Ring-detected tees are high-precision hollow-rect seeds; with
# them Heritage recovers to 14/18@25 while DashsTrack stays perfect.
# The metric itself (F-beta 0.5, precision-favored border agreement)
# is unchanged — it was right all along; only seeding moved.
# ------------------------------------------------------------

_ring_comps = []
for _t in tees:
    if _t["tier"] != "ring":
        continue
    _cands = [c for c in bright_components
              if np.hypot(c.cx - _t["cx"], c.cy - _t["cy"]) <= 15]
    if _cands:
        _ring_comps.append(min(
            _cands,
            key=lambda c: np.hypot(c.cx - _t["cx"], c.cy - _t["cy"])))
_ring_comps = list({c.label: c for c in _ring_comps}.values())
if len(_ring_comps) < SEED_COUNT:
    raise ValueError("not enough ring-tier tees to seed the border metric")
_med_major = np.median([c.major for c in _ring_comps])
_med_minor = np.median([c.minor for c in _ring_comps])
RING_SEEDS = sorted(
    _ring_comps,
    key=lambda c: max(abs(np.log(c.major / _med_major)),
                      abs(np.log(c.minor / _med_minor))))[:SEED_COUNT]
modal_major = _med_major
modal_minor = _med_minor
seeds = RING_SEEDS

canon_scale = CANON_MAJOR_SPAN / modal_major

seed_masks = [
    canonical_component_mask(c, canon_scale)
    for c in seeds
]


# ------------------------------------------------------------
# 4. Learn fuzzy consensus border from seed tee candidates
# ------------------------------------------------------------

seed_likelihoods = []

for mask in seed_masks:
    dist = distance_to_foreground(mask)
    seed_likelihoods.append(
        fuzzy_support(dist)
    )

border_likelihood = np.mean(
    seed_likelihoods,
    axis=0
)

# Rough consensus band:
# pixel should be reasonably supported by >= 2-ish seeds.
template_border = (
    border_likelihood >= 0.50
).astype(np.uint8)

template_distance = distance_to_foreground(
    template_border
)

template_pixels = template_border.astype(bool)


# ------------------------------------------------------------
# 5. Score candidate against template
# ------------------------------------------------------------

def score_candidate(c):
    raw = canonical_component_mask(
        c,
        canon_scale
    )

    best = None

    for dy in SHIFT_VALUES:
        for dx in SHIFT_VALUES:

            candidate = shift_mask(
                raw,
                dx,
                dy
            )

            observed = candidate.astype(bool)

            if observed.sum() == 0:
                continue

            # ------------------------------------------------
            # PRECISION / EXPLAINED
            #
            # "Of the bright pixels I actually observed,
            #  how many lie where tee border is expected?"
            # ------------------------------------------------

            observed_dist = template_distance[observed]

            explained = np.mean(
                fuzzy_support(observed_dist)
            )

            # ------------------------------------------------
            # RECALL / COVERAGE
            #
            # "How much expected tee border has surviving
            #  observed support?"
            #
            # Missing border hurts, but substantially less
            # than unexplained observed pixels.
            # ------------------------------------------------

            candidate_dist = distance_to_foreground(
                candidate
            )

            expected_dist = candidate_dist[
                template_pixels
            ]

            coverage = np.mean(
                fuzzy_support(expected_dist)
            )

            score = fbeta(
                explained,
                coverage,
                beta=F_BETA
            )

            row = {
                "component": c,
                "score": float(score),
                "explained": float(explained),
                "coverage": float(coverage),
                "dx": dx,
                "dy": dy,
                "mask": candidate
            }

            if (
                best is None
                or row["score"] > best["score"]
            ):
                best = row

    return best




# ------------------------------------------------------------
# Rank the full candidate universe (bright components minus badges,
# sprites, badge digits) by the deliberate border metric.
# ------------------------------------------------------------

_sprite_labels = set()
for sp in sprites:
    roi = bright_labels[sp["y"]:sp["y"] + TH, sp["x"]:sp["x"] + TW]
    for lb in np.unique(roi[template.astype(bool)]):
        if lb > 0:
            _sprite_labels.add(int(lb))
_badge_labels = {c.label for c in badges}
candidate_universe = [
    c for c in bright_components
    if c.label not in _badge_labels
    and c.label not in _sprite_labels
    and not inside_badge(c.cx, c.cy)
]

results = []
for c in candidate_universe:
    r = score_candidate(c)
    if r is not None:
        results.append(r)
results.sort(key=lambda r: r["score"], reverse=True)

print(f"border-ranked {len(results)} candidates "
      f"(seeds: {[s.label for s in RING_SEEDS]})")
print("top 25:")
for i, r in enumerate(results[:25], 1):
    c = r["component"]
    print(f"  {i:2d}. label={c.label:3d} ({c.cx:6.1f},{c.cy:6.1f}) "
          f"score={r['score']:.3f} explained={r['explained']:.3f} "
          f"coverage={r['coverage']:.3f}")

# Attach border score to the cell-3 tee list (verification channel).
for t in tees:
    t["border_score"] = None
    for r in results:
        c = r["component"]
        if np.hypot(c.cx - t["cx"], c.cy - t["cy"]) <= 15:
            t["border_score"] = r["score"]
            break
print("\ncell-3 tees with border verification:")
for i, t in enumerate(tees):
    bs = t["border_score"]
    print(f"  tee{i:2d} tier={t['tier']:9s} "
          f"border={'%.3f' % bs if bs is not None else '  n/a'}")


In [ ]:
# @title
# ============================================================
# Invariant cross-checks + final P1 picture
#
# Measured invariants (dev truth):
#   * the tee long axis points at its own badge:
#     true badge median 1.1 deg, p90 2.65; other badges median 38 deg;
#   * the badge projects onto tee->basket at 0.17-0.54 (median ~0.5),
#     always before any bend;
#   * straight holes: tee/badge/basket collinear to <= 1.4 deg.
#
# Here (P1 scope: no pairing) each ring-tier tee reports its
# best-aligned badge and the alignment error — a <= ~6 deg alignment is
# strong evidence of the tee's own hole; ~38 deg is chance.
# ============================================================


def ang_diff(a_deg, b_deg):
    d = abs((a_deg - b_deg) % 180)
    return min(d, 180 - d)


print("ring-tee -> best-aligned badge:")
alignments = []
for i, t in enumerate(tees):
    if t["angle"] is None:
        continue
    best = None
    for j, b in enumerate(badges):
        bearing = np.degrees(np.arctan2(b.cy - t["cy"], b.cx - t["cx"])) % 180
        d = ang_diff(t["angle"], bearing)
        if best is None or d < best[0]:
            best = (d, j, b)
    if best:
        alignments.append(best[0])
        print(f"  tee{i:2d} ({t['cx']:6.1f},{t['cy']:6.1f}) "
              f"axis={t['angle']:5.1f}  ->  Bdg{best[1] + 1} "
              f"err={best[0]:4.1f} deg "
              f"{'OK' if best[0] <= 6 else '??'}")
if alignments:
    print(f"aligned<=6deg: {sum(1 for a in alignments if a <= 6)}"
          f"/{len(alignments)}")

fig, ax = plt.subplots(figsize=(15, 15))
ax.imshow(rgb)
for i, c in enumerate(badges, 1):
    rect = plt.Rectangle((c.bbox_x, c.bbox_y), c.bbox_w, c.bbox_h,
                         fill=False, linewidth=2, edgecolor="red")
    ax.add_patch(rect)
    ax.text(c.cx, c.cy, f"Bdg{i}", fontsize=7, ha="center", va="center",
            bbox=dict(facecolor="white", alpha=0.72, edgecolor="none", pad=1))
for i, sp in enumerate(sprites, 1):
    rect = plt.Rectangle((sp["x"], sp["y"]), TW, TH, fill=False,
                         linewidth=2, edgecolor="blue")
    ax.add_patch(rect)
    ax.plot(sp["tip_x"], sp["tip_y"], "b+", markersize=9)
    ax.text(sp["cx"], sp["y"] - 6, f"Bkt{i} {sp['score']:.2f}", fontsize=7,
            ha="center",
            bbox=dict(facecolor="white", alpha=0.72, edgecolor="none", pad=1))
for i, t in enumerate(tees, 1):
    color = "lime" if t["tier"] == "ring" else "orange"
    ax.plot(t["cx"], t["cy"], "o", color=color, markersize=6,
            markerfacecolor="none")
    if t["angle"] is not None:
        th = np.radians(t["angle"])
        L = 26
        ax.plot([t["cx"] - L * np.cos(th), t["cx"] + L * np.cos(th)],
                [t["cy"] - L * np.sin(th), t["cy"] + L * np.sin(th)],
                color=color, linewidth=1)
for d in diamonds:
    ax.plot(d["cx"], d["cy"], "x", color="cyan", markersize=6)
ax.set_title(
    "NuThing P1 Lab v2 (render-identity) | "
    f"badges={len(badges)} baskets={len(sprites)} tees={len(tees)} "
    f"diamonds={len(diamonds)}")
ax.axis("off")
plt.show()

print("\nTIMINGS")
for k, t in TIMINGS.items():
    print(f"  {k:16s} {t * 1000:7.1f} ms")
